# Antardrishti — the optical flood rule, scored like the radar rule

**Question.** `/analyze` now answers floods with Sentinel-2 as well as Sentinel-1, and both emit the same evidence quantities. But the optical rule (MNDWI above −0.15) carries **no accuracy figure**, because the only number we had — IoU 0.466 — was measured for *surface water against WorldCover permanent water*, a different task. This notebook scores the optical flood rule on **the same 446 Sen1Floods11 chips, on the same pixels**, as the radar rule that ships (mean(VV,VH) below −20 dB, IoU 0.489).

**What would count as an answer.** One validation block for `backend/detection/optical.py`, measured at the threshold that actually ships, plus a head-to-head table in which neither sensor is scored on ground the other was excused from.

**Read before believing any number below.**

1. **The labels lean optical.** Sen1Floods11 labels were drawn from Sentinel-2 NDVI/MNDWI and hand corrected. Scoring MNDWI against labels partly made from MNDWI is partly circular, so the optical score is biased *upward*. A radar win here is strong evidence; an optical win is weak evidence.
2. **Cloud is excused.** Pixels the labellers could not see are −1 and are never scored. Optical is never penalised here for cloud — which is what decides between the sensors in a monsoon. This measures accuracy where the ground is visible, not usefulness.
3. **Product level.** These chips are Level‑1C (top of atmosphere). The backend uses Level‑2A (surface reflectance). The best threshold here may sit slightly off the best in production.

In [13]:
# One-time install. rasterio ships Windows wheels, so there is no compiler step.
!pip install rasterio numpy


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Data

The same archive as notebook 04 — it already contains the Sentinel-2 chips (`*_S2Hand.tif`) alongside the radar ones. If you ran notebook 04, point `LOCAL` at the same folder and there is nothing to download.

Otherwise, once, in a terminal:

```
pip install huggingface_hub
hf download blumenstiel/Sen1Floods11 sen1floods11_v1_1.tar.gz --repo-type dataset --local-dir .
tar -xf sen1floods11_v1_1.tar.gz
```

In [14]:
%pip install huggingface_hub

import tarfile
from huggingface_hub import hf_hub_download

TARGET = r"C:\sen1floods11"

archive = hf_hub_download(
    "blumenstiel/Sen1Floods11", "sen1floods11_v1_1.tar.gz",
    repo_type="dataset", local_dir=TARGET,
)
with tarfile.open(archive) as tar:
    tar.extractall(TARGET)

print("done:", TARGET)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


done: C:\sen1floods11


In [15]:
# Wherever you extracted the archive. Windows paths need the r"" prefix.
LOCAL = r"C:\sen1floods11"

from pathlib import Path

root = Path(LOCAL)
if not root.exists():
    raise SystemExit(f"{root} does not exist - see the download commands above")

s1_files = {p.name.replace("_S1Hand.tif", ""): p for p in root.rglob("*_S1Hand.tif")}
s2_files = {p.name.replace("_S2Hand.tif", ""): p for p in root.rglob("*_S2Hand.tif")}
labels   = {p.name.replace("_LabelHand.tif", ""): p for p in root.rglob("*_LabelHand.tif")}

# Only chips that have all three. Scoring optical on chips radar never saw, or
# the reverse, would make the two columns of the final table describe
# different samples.
keys = sorted(set(s1_files) & set(s2_files) & set(labels))

print(f"S1 chips: {len(s1_files)}   S2 chips: {len(s2_files)}   labels: {len(labels)}")
print(f"chips with all three: {len(keys)}")
if not keys:
    raise SystemExit("no complete triples - check LOCAL; the files may be one level deeper")

S1 chips: 446   S2 chips: 446   labels: 446
chips with all three: 446


## 2. Our own code

`evaluation/optical.py` mirrors the backend's optical rule, `evaluation/thresholds.py` mirrors the radar rule. Both are unit-tested in `backend/tests/` — including a test that the optical threshold scored here equals the one in `backend/detection/optical.py`, so this notebook cannot quietly measure a rule nobody runs.

In [16]:
import sys
from pathlib import Path

EVALUATION = Path.cwd().resolve().parent.parent / "evaluation"
if not EVALUATION.exists():
    raise SystemExit(f"{EVALUATION} not found - start Jupyter from backend/notebooks/")
sys.path.insert(0, str(EVALUATION))

import json
import numpy as np
import rasterio

import optical as O
import thresholds as T
from metrics import evaluate, aggregate, format_table

print(f"optical rule: MNDWI > {O.SHIPPED_MNDWI}")
print(f"radar rule:   {T.SHIPPED_METHOD} < {T.SHIPPED_DB} dB")

optical rule: MNDWI > -0.15
radar rule:   mean(VV,VH) < -20.0 dB


In [17]:
# Kept per chip: MNDWI, mean(VV,VH) in dB, and the labels restricted to pixels
# BOTH sensors have data for. float32, so all 446 chips fit in about 1 GB.
# Short on RAM? Set LIMIT = 100 for a first pass; the conclusions hold.
LIMIT = None
NODATA_FLOOR_DB = -50.0

def load_s1(path):
    with rasterio.open(path) as src:
        vv = src.read(1).astype(np.float64)
        vh = src.read(2).astype(np.float64)
    vv[vv < NODATA_FLOOR_DB] = np.nan
    vh[vh < NODATA_FLOOR_DB] = np.nan
    return vv, vh

def load_label(path):
    with rasterio.open(path) as src:
        return src.read(1).astype(np.int8)

loaded, skipped = [], 0
for key in (keys[:LIMIT] if LIMIT else keys):
    try:
        vv, vh = load_s1(s1_files[key])
        green, swir1 = O.load_s2(s2_files[key])
        truth = load_label(labels[key])
    except Exception as exc:
        print(f"skip {key}: {exc}")
        skipped += 1
        continue

    index = O.mndwi(green, swir1)
    with np.errstate(invalid="ignore"):
        sar_mean = (vv + vh) / 2.0

    valid = O.common_valid(truth, index, sar_mean)
    if valid.sum() == 0:
        skipped += 1
        continue

    loaded.append({
        "key": key,
        "mndwi": index.astype(np.float32),
        "sar": sar_mean.astype(np.float32),
        "truth": O.restrict(truth, valid),
        "labelled": int((truth != O.LABEL_NODATA).sum()),
        "common": int(valid.sum()),
    })

labelled = sum(c["labelled"] for c in loaded)
common = sum(c["common"] for c in loaded)
print(f"{len(loaded)} chips loaded, {skipped} skipped")
print(f"labelled pixels: {labelled:,}   scored by both sensors: {common:,} "
      f"({common / labelled:.1%})")
print("\nIf the second number is much smaller than the first, one sensor was")
print("missing data over a lot of labelled ground. Those pixels are left out of")
print("BOTH columns below, which is what keeps the comparison fair.")

440 chips loaded, 6 skipped
labelled pixels: 100,933,288   scored by both sensors: 100,800,229 (99.9%)

If the second number is much smaller than the first, one sensor was
missing data over a lot of labelled ground. Those pixels are left out of
BOTH columns below, which is what keeps the comparison fair.


## 3. Sweep the optical threshold

−0.15 was chosen for *surface water against WorldCover*. It may not be best for floods. The sweep says where the optimum is — but the validation block written at the end describes **−0.15, the value that ships**, because a score only applies to the threshold it was measured at.

In [18]:
sweep = {}
for t in np.round(np.arange(-0.50, 0.51, 0.05), 2):
    sweep[float(t)] = aggregate([
        evaluate(np.isfinite(c["mndwi"]) & (c["mndwi"] > t), c["truth"])
        for c in loaded
    ])

print(f"{'MNDWI >':>8}{'IoU':>8}{'prec':>8}{'recall':>8}")
print("-" * 32)
best = None
for t, agg in sweep.items():
    m = agg["micro"]
    marker = "  <- ships" if abs(t - O.SHIPPED_MNDWI) < 1e-9 else ""
    print(f"{t:>8.2f}{m['iou']:>8.3f}{m['precision']:>8.3f}{m['recall']:>8.3f}{marker}")
    if best is None or m["iou"] > best[1]:
        best = (t, m["iou"])

BEST_MNDWI = best[0]
print(f"\nBest: MNDWI > {BEST_MNDWI:.2f} at IoU {best[1]:.3f}")

 MNDWI >     IoU    prec  recall
--------------------------------
   -0.50   0.106   0.106   1.000
   -0.45   0.108   0.108   1.000
   -0.40   0.116   0.116   1.000
   -0.35   0.132   0.132   0.999
   -0.30   0.167   0.167   0.998
   -0.25   0.224   0.224   0.995
   -0.20   0.294   0.295   0.992
   -0.15   0.367   0.369   0.987  <- ships
   -0.10   0.444   0.448   0.982
   -0.05   0.520   0.527   0.975
   -0.00   0.596   0.609   0.964
    0.05   0.663   0.689   0.946
    0.10   0.716   0.763   0.920
    0.15   0.743   0.829   0.878
    0.20   0.736   0.874   0.823
    0.25   0.725   0.911   0.781
    0.30   0.701   0.938   0.735
    0.35   0.667   0.958   0.688
    0.40   0.615   0.969   0.627
    0.45   0.552   0.974   0.560
    0.50   0.483   0.976   0.489

Best: MNDWI > 0.15 at IoU 0.743


## 4. Head to head, on the same pixels

Both rules as they ship, plus the optical optimum from the sweep. Every row is scored on exactly the same pixels.

In [19]:
def score(predict):
    return aggregate([evaluate(predict(c), c["truth"]) for c in loaded])

RULES = {
    f"Radar  mean(VV,VH) < {T.SHIPPED_DB} dB (ships)":
        lambda c: np.isfinite(c["sar"]) & (c["sar"] < T.SHIPPED_DB),
    f"Optical MNDWI > {O.SHIPPED_MNDWI} (ships)":
        lambda c: np.isfinite(c["mndwi"]) & (c["mndwi"] > O.SHIPPED_MNDWI),
    f"Optical MNDWI > {BEST_MNDWI} (sweep best)":
        lambda c: np.isfinite(c["mndwi"]) & (c["mndwi"] > BEST_MNDWI),
}
results = {name: score(rule) for name, rule in RULES.items()}

print(format_table(results, "Sen1Floods11, common pixels only"))
print()
print("Reminder: the optical rows are scored against labels partly drawn from")
print("MNDWI, and cloud never counts against them. See the caveats at the top.")

Sen1Floods11, common pixels only
method                           IoU      F1    prec  recall   chips
--------------------------------------------------------------------
Radar  mean(VV,VH) < -20.0 dB (ships)   0.491   0.659   0.768   0.577     440
Optical MNDWI > -0.15 (ships)   0.367   0.537   0.369   0.987     440
Optical MNDWI > 0.15 (sweep best)   0.743   0.853   0.829   0.878     440

(micro-averaged: all pixels pooled)

Reminder: the optical rows are scored against labels partly drawn from
MNDWI, and cloud never counts against them. See the caveats at the top.


## 5. Do they fail in the same places?

If radar and optical miss *different* water, each covers the other's gap and there is a case for using both. If they miss the *same* water, a second sensor adds nothing but cost.

In [20]:
radar_rule = list(RULES.values())[0]
optical_rule = list(RULES.values())[1]

counts = {"both right": 0, "only radar right": 0, "only optical right": 0, "both wrong": 0}
water_counts = dict.fromkeys(counts, 0)

for c in loaded:
    scored = c["truth"] != O.LABEL_NODATA
    truth = c["truth"][scored].astype(bool)
    r_ok = radar_rule(c)[scored] == truth
    o_ok = optical_rule(c)[scored] == truth
    for name, sel in (("both right", r_ok & o_ok), ("only radar right", r_ok & ~o_ok),
                      ("only optical right", ~r_ok & o_ok), ("both wrong", ~r_ok & ~o_ok)):
        counts[name] += int(sel.sum())
        water_counts[name] += int((sel & truth).sum())

total, water_total = sum(counts.values()), sum(water_counts.values())
print(f"{'':<20}{'all pixels':>14}{'water pixels':>16}")
print("-" * 50)
for name in counts:
    print(f"{name:<20}{counts[name] / total:>14.1%}{water_counts[name] / water_total:>16.1%}")

print("\nThe water column is the one that matters for recall: 'only optical right'")
print("there is flood water radar missed and optical found - and vice versa.")

                        all pixels    water pixels
--------------------------------------------------
both right                   76.4%           57.6%
only radar right             17.2%            0.1%
only optical right            5.6%           41.1%
both wrong                    0.7%            1.2%

The water column is the one that matters for recall: 'only optical right'
there is flood water radar missed and optical found - and vice versa.


In [21]:
shipped_optical = results[f"Optical MNDWI > {O.SHIPPED_MNDWI} (ships)"]["micro"]
shipped_radar = results[f"Radar  mean(VV,VH) < {T.SHIPPED_DB} dB (ships)"]["micro"]

validation_block = {
    "dataset": f"Sen1Floods11 HandLabeled ({len(loaded)} chips, S2 L1C, pixels common with S1)",
    "iou": round(shipped_optical["iou"], 3),
    "precision": round(shipped_optical["precision"], 3),
    "recall": round(shipped_optical["recall"], 3),
}

payload = {
    "question": "How accurate is the optical flood rule, scored like the radar rule?",
    "chips_scored": len(loaded),
    "labelled_pixels": labelled,
    "common_pixels": common,
    "shipped_optical_threshold": O.SHIPPED_MNDWI,
    "best_optical_threshold": BEST_MNDWI,
    "sweep": {str(t): agg["micro"] for t, agg in sweep.items()},
    "head_to_head": {name: {"micro": a["micro"], "macro": a["macro"]}
                     for name, a in results.items()},
    "agreement": {"all_pixels": counts, "water_pixels": water_counts},
    "radar_on_common_pixels": {k: shipped_radar[k] for k in ("iou", "precision", "recall")},
    "validation_block_for_backend": validation_block,
    "label_caveat": O.LABEL_CAVEAT,
}

out = EVALUATION / "optical_flood_results.json"
with out.open("w", encoding="utf-8") as fh:
    json.dump(payload, fh, indent=2)

print(f"written to {out}\n")
print("Paste this into backend/detection/optical.py as VALIDATION:\n")
print(json.dumps(validation_block, indent=4))

written to C:\Users\punit\OneDrive\Desktop\Group_project\application\antardrishti\evaluation\optical_flood_results.json

Paste this into backend/detection/optical.py as VALIDATION:

{
    "dataset": "Sen1Floods11 HandLabeled (440 chips, S2 L1C, pixels common with S1)",
    "iou": 0.367,
    "precision": 0.369,
    "recall": 0.987
}


## What to do with the numbers

**Fill in `VALIDATION`.** Paste the printed block into `backend/detection/optical.py`. That is the only change needed: confidence, the accuracy caveat in every report, the map styling and the PDF all read it from there, and `test_once_measured_the_validation_flows_through_like_sars` already checks that path. The "not yet scored" note disappears by itself.

**Note the radar row is on common pixels.** It will not be exactly 0.489 — that figure is over every pixel radar could see; this one is over pixels *both* sensors could see. Quote this row, not 0.489, in any sentence that compares the two sensors.

**If the sweep optimum is well away from −0.15.** Do not change `water_extent` in `detection/surface.py` — that threshold was measured for a different task and is right for it. Instead give `detection/optical.py` its own `THRESHOLD`, update `SHIPPED_MNDWI` in `evaluation/optical.py` to match, rerun this notebook, and paste the new block. `test_the_scored_threshold_is_the_shipped_threshold` fails until both are changed, which is the point.

**How to write it up.** Given the label bias, the defensible sentence is conditional: "On pixels both sensors could see, and against labels drawn partly from optical indices, MNDWI scores X against radar's Y." If radar still wins, that is a strong result. If optical wins, say it wins *where it can see*, and put the cloud-coverage figures from a monsoon run next to it — that is where the sensors actually differ.